# Phase 2 TFIM-QRC v1.5: Full Topology + Virtual Nodes

This notebook tests a controlled upgrade of the best v1 static QRC branch, not the failed feedback branch.

Changes relative to v1 best:

- preserve PCA-6, 6 qubits, 6 anchors, ZXZZ, disorder=0.20, ridge alpha=3000;
- change Hamiltonian topology from chain to fully connected ZZ interactions;
- collect virtual-node observables after intermediate Trotter steps inside each anchor;
- sweep reservoir timescale via `evolution_time`.

Main question: does richer TFIM mixing + virtual-node temporal multiplexing improve reservoir diagnostics and out-of-sample performance without changing the whole modeling framework?

In [1]:
from pathlib import Path
import os

if Path.cwd().name == "notebooks":
    os.chdir("..")

import pandas as pd

from qpitome_qrc.data.features import FEATURE_COLUMNS
from qpitome_qrc.data.loaders import load_phase2_volatility_data
from qpitome_qrc.data.pca import fit_transform_pca_splits_train_only
from qpitome_qrc.data.splits import chronological_tabular_split
from qpitome_qrc.qrc.tfim_reservoir import (
    TFIMQRCConfig,
    diagnose_reservoir_feature_splits,
    fit_tfim_qrc_regressor,
    make_qrc_sequence_splits,
    summarize_qrc_result,
)

## 1. Data and PCA-6 sequence windows

In [2]:
target = "future_rv_20d"

df = load_phase2_volatility_data()
splits = chronological_tabular_split(df)

pca6 = fit_transform_pca_splits_train_only(
    splits,
    feature_columns=FEATURE_COLUMNS,
    target_columns=[target],
    n_components=6,
    prefix="pca6",
)

sequence_splits_6 = make_qrc_sequence_splits(
    pca6.splits,
    feature_columns=pca6.feature_columns,
    target_column=target,
    lookback_days=40,
)

display(pca6.explained_variance)
print({name: (X.shape, y.shape) for name, (X, y, dates) in sequence_splits_6.items()})

,component,explained_variance_ratio,cumulative_explained_variance
0,1,0.408573,0.408573
1,2,0.112421,0.520994
2,3,0.090849,0.611843
3,4,0.073259,0.685102
4,5,0.064206,0.749308
5,6,0.055584,0.804892


{'train': ((5420, 40, 6), (5420,)), 'val': ((1219, 40, 6), (1219,)), 'test': ((1019, 40, 6), (1019,))}


## 2. v1.5 timescale sweep

Fixed architecture:

```text
6q / PCA-6 / 6 anchors / ZXZZ
topology = full
trotter_steps_per_anchor = 3
virtual_nodes_per_anchor = 3
disorder_strength = 0.20
ridge_alpha = 3000
```

Sweep only `evolution_time`.

In [3]:
v15_rows = []
v15_diag_rows = []
v15_results = {}

for evolution_time in [0.5, 1.5, 3.0, 5.0]:
    config = TFIMQRCConfig(
        qubits=6,
        pca_components=6,
        lookback_days=40,
        anchor_count=6,
        anchor_policy="even",
        observable_mode="zxzz",
        collect_anchor_features=True,
        topology="full",
        trotter_steps_per_anchor=3,
        virtual_nodes_per_anchor=3,
        coupling_scale=0.7,
        transverse_field=0.5,
        evolution_time=float(evolution_time),
        angle_max=3.141592653589793 / 2,
        ridge_alpha=3000.0,
        target_transform="log",
        seed=42,
        use_disorder=True,
        disorder_strength=0.20,
    )

    run_name = f"v15_full_virtualnodes_evolution_{evolution_time}"
    print(f"Running {run_name}")

    result = fit_tfim_qrc_regressor(
        sequence_splits_6,
        config=config,
        target=target,
        verbose=True,
    )

    v15_results[run_name] = result

    row = summarize_qrc_result(result)
    row["run_name"] = run_name
    v15_rows.append(row)

    _, y_train, _ = sequence_splits_6["train"]
    _, y_val, _ = sequence_splits_6["val"]
    _, y_test, _ = sequence_splits_6["test"]

    diag = diagnose_reservoir_feature_splits(
        result.train_features,
        result.val_features,
        result.test_features,
        y_train,
        y_val,
        y_test,
    )
    diag.insert(0, "run_name", run_name)
    v15_diag_rows.append(diag)

v15_table = pd.DataFrame(v15_rows)
v15_diagnostics = pd.concat(v15_diag_rows, ignore_index=True)

Running v15_full_virtualnodes_evolution_0.5
QRC sample 0/5420
QRC sample 250/5420
QRC sample 500/5420
QRC sample 750/5420
QRC sample 1000/5420
QRC sample 1250/5420
QRC sample 1500/5420
QRC sample 1750/5420
QRC sample 2000/5420
QRC sample 2250/5420
QRC sample 2500/5420
QRC sample 2750/5420
QRC sample 3000/5420
QRC sample 3250/5420
QRC sample 3500/5420
QRC sample 3750/5420
QRC sample 4000/5420
QRC sample 4250/5420
QRC sample 4500/5420
QRC sample 4750/5420
QRC sample 5000/5420
QRC sample 5250/5420
QRC sample 0/1219
QRC sample 250/1219
QRC sample 500/1219
QRC sample 750/1219
QRC sample 1000/1219
QRC sample 0/1019
QRC sample 250/1019
QRC sample 500/1019
QRC sample 750/1019
QRC sample 1000/1019
Running v15_full_virtualnodes_evolution_1.5
QRC sample 0/5420
QRC sample 250/5420
QRC sample 500/5420
QRC sample 750/5420
QRC sample 1000/5420
QRC sample 1250/5420
QRC sample 1500/5420
QRC sample 1750/5420
QRC sample 2000/5420
QRC sample 2250/5420
QRC sample 2500/5420
QRC sample 2750/5420
QRC sample 3

## 3. Metrics

In [4]:
metric_cols = [
    "run_name",
    "topology",
    "trotter_steps_per_anchor",
    "virtual_nodes_per_anchor",
    "evolution_time",
    "n_reservoir_features",
    "train_rmse",
    "val_rmse",
    "test_rmse",
    "train_qlike",
    "val_qlike",
    "test_qlike",
    "train_mz_r2",
    "val_mz_r2",
    "test_mz_r2",
]

v15_table[metric_cols].sort_values("test_rmse")

,run_name,topology,trotter_steps_per_anchor,virtual_nodes_per_anchor,evolution_time,n_reservoir_features,train_rmse,val_rmse,test_rmse,train_qlike,val_qlike,test_qlike,train_mz_r2,val_mz_r2,test_mz_r2
0,v15_full_virtualnodes_evolution_0.5,full,3,3,0.5,306,0.079786,0.061480,0.102084,-2.549892,-3.013282,-2.033862,0.394225,0.072823,0.080576
2,v15_full_virtualnodes_evolution_3.0,full,3,3,3.0,306,0.084684,0.067016,0.104984,-2.484217,-2.939455,-1.797534,0.314153,0.013008,0.045103
1,v15_full_virtualnodes_evolution_1.5,full,3,3,1.5,306,0.084063,0.064791,0.106082,-2.489027,-2.963931,-1.788168,0.324486,0.026058,0.037069
3,v15_full_virtualnodes_evolution_5.0,full,3,3,5.0,306,0.083743,0.070979,0.106726,-2.499064,-2.884122,-1.663998,0.325334,0.003331,0.032103


## 4. Diagnostics

In [5]:
diagnostic_cols = [
    "run_name",
    "split",
    "n_samples",
    "n_features",
    "near_constant_features",
    "feature_std_min",
    "feature_std_median",
    "feature_std_max",
    "effective_rank",
    "condition_number",
    "mean_abs_feature_target_corr",
    "max_abs_feature_target_corr",
    "mean_abs_shift_vs_train",
    "max_abs_shift_vs_train",
]

v15_diagnostics[diagnostic_cols]

,run_name,split,n_samples,n_features,near_constant_features,feature_std_min,feature_std_median,feature_std_max,effective_rank,condition_number,mean_abs_feature_target_corr,max_abs_feature_target_corr,mean_abs_shift_vs_train,max_abs_shift_vs_train
0,v15_full_virtualnodes_evolution_0.5,train,5420,306,0,0.145687,0.263086,0.457878,127.761107,6177.332033,0.138009,0.470035,0.000000,0.000000
1,v15_full_virtualnodes_evolution_0.5,val,1219,306,0,0.117309,0.242968,0.422090,118.457550,14816.513580,0.069288,0.252895,0.160010,0.974529
2,v15_full_virtualnodes_evolution_0.5,test,1019,306,0,0.141476,0.265688,0.453292,120.620387,10776.722249,0.111833,0.298595,0.164266,0.743272
3,v15_full_virtualnodes_evolution_1.5,train,5420,306,0,0.112359,0.205887,0.465406,184.016179,447.097421,0.124292,0.403006,0.000000,0.000000
4,v15_full_virtualnodes_evolution_1.5,val,1219,306,0,0.093407,0.182444,0.404843,175.464585,756.086974,0.059992,0.181159,0.161581,0.897388
5,v15_full_virtualnodes_evolution_1.5,test,1019,306,0,0.105369,0.199642,0.453680,175.349636,643.693992,0.132182,0.302136,0.158766,0.711573
6,v15_full_virtualnodes_evolution_3.0,train,5420,306,0,0.062239,0.127959,0.298027,251.862804,50.212635,0.070267,0.366333,0.000000,0.000000
7,v15_full_virtualnodes_evolution_3.0,val,1219,306,0,0.058062,0.123458,0.285570,235.363611,108.478853,0.056156,0.200085,0.155322,1.025927
8,v15_full_virtualnodes_evolution_3.0,test,1019,306,0,0.059438,0.126554,0.281383,239.596627,87.059988,0.082523,0.300092,0.122964,0.642612
9,v15_full_virtualnodes_evolution_5.0,train,5420,306,0,0.026528,0.122729,0.372573,264.629560,35.458650,0.073141,0.401537,0.000000,0.000000


## 5. Save outputs

In [6]:
out_dir = Path("results/tables")
out_dir.mkdir(parents=True, exist_ok=True)

v15_table.to_csv(out_dir / "phase2_tfim_qrc_v15_full_virtualnodes_probe.csv", index=False)
v15_diagnostics.to_csv(out_dir / "phase2_tfim_qrc_v15_full_virtualnodes_diagnostics.csv", index=False)

print("Saved v1.5 full-topology virtual-node outputs to", out_dir)

Saved v1.5 full-topology virtual-node outputs to results/tables


## 6. Interpretation rule

Compare against current best v1:

```text
6q / PCA-6 / ZXZZ / snapshots / chain / disorder=0.20 / alpha=3000
test RMSE  = 0.102618
test QLIKE = -1.942716
test MZ R² = 0.072134
```

This test should be judged first by reservoir diagnostics, then by validation/test metrics. If all evolution times degrade feature stability and test performance, stop this architecture branch. If one timescale improves diagnostics and validation performance, run a smaller second sweep around that timescale.